Projeto: Merca Data Platform

Squad: 2 | Streaming em Tempo Real

Polling de diretórios para detectar novos snapshots 

In [0]:
import os
import io
import time
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

load_dotenv()

ADLS_CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
ADLS_TENANT_ID       = os.getenv("ADLS_TENANT_ID")
ADLS_CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
ADLS_STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT")
ADLS_CONTAINER       = os.getenv("ADLS_CONTAINER")

# Autentica no ADLS
credential = ClientSecretCredential(
    tenant_id     = ADLS_TENANT_ID,
    client_id     = ADLS_CLIENT_ID,
    client_secret = ADLS_CLIENT_SECRET
)

service_client = DataLakeServiceClient(
    account_url = f"https://{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential  = credential
)

container_client = service_client.get_file_system_client(ADLS_CONTAINER)

print("Conexão com ADLS estabelecida!")

In [0]:
# Configurações
BASE_PATH      = "vendas_raw"
TABELAS        = [
    "ecommerce_categorias",
    "ecommerce_itens_pedido",
    "ecommerce_produtos"
]
INTERVALO_SEG  = 30   # verifica a cada 30 segundos
MAX_CICLOS     = 10   # número máximo de verificações

print(f"⚙️  Configurações do Polling:")
print(f"   Base Path    : {BASE_PATH}")
print(f"   Tabelas      : {TABELAS}")
print(f"   Intervalo    : {INTERVALO_SEG}s")
print(f"   Máx. Ciclos  : {MAX_CICLOS}")

In [0]:
def listar_snapshots(container_client, base_path):
    """Lista todas as pastas de snapshot disponíveis no lake."""
    snapshots = set()

    paths = container_client.get_paths(path=base_path, recursive=True)

    for item in paths:
        # Padrão: vendas_raw/YYYY/MM/DD/HHMMSS
        partes = item.name.replace(base_path + "/", "").split("/")
        if len(partes) == 4 and item.is_directory:
            snapshot_id = "/".join(partes)  # ex: 2026/04/27/225222
            snapshots.add(snapshot_id)

    return snapshots


def ler_tabela(container_client, base_path, snapshot_id, tabela):
    """Lê um arquivo parquet de um snapshot específico."""
    file_path   = f"{base_path}/{snapshot_id}/{tabela}.parquet"
    file_client = container_client.get_file_client(file_path)

    download   = file_client.download_file()
    bytes_data = download.readall()
    pdf        = pd.read_parquet(io.BytesIO(bytes_data))

    return spark.createDataFrame(pdf)


def processar_snapshot(container_client, base_path, snapshot_id, tabelas):
    """Processa todas as tabelas de um snapshot novo."""
    print(f"\n  🔄 Processando snapshot: {snapshot_id}")

    for tabela in tabelas:
        try:
            df = ler_tabela(container_client, base_path, snapshot_id, tabela)
            print(f"  ✅ {tabela} → {df.count()} linhas")
            # Aqui futuramente salvaremos no SQL Server
        except Exception as e:
            print(f"  Erro em {tabela}: {str(e)}")


print(" Funções auxiliares carregadas!")

In [0]:
print("=" * 55)
print("   INICIANDO POLLING DE DIRETÓRIOS")
print("=" * 55)

# Carrega snapshots já existentes como "já processados"
snapshots_processados = listar_snapshots(container_client, BASE_PATH)
print(f"\nPacotes {len(snapshots_processados)} snapshot(s) existentes ignorados.")
print(f" Aguardando novos snapshots...\n")

ciclo = 0

while ciclo < MAX_CICLOS:
    ciclo += 1
    agora = datetime.now().strftime("%H:%M:%S")

    print(f" [{agora}] Ciclo {ciclo}/{MAX_CICLOS} — verificando...")

    # Lista snapshots atuais
    snapshots_atuais = listar_snapshots(container_client, BASE_PATH)

    # Detecta novos snapshots
    novos = snapshots_atuais - snapshots_processados

    if novos:
        print(f"   {len(novos)} novo(s) snapshot(s) detectado(s)!")
        for snapshot_id in sorted(novos):
            processar_snapshot(container_client, BASE_PATH, snapshot_id, TABELAS)
            snapshots_processados.add(snapshot_id)
    else:
        print(f"  ⏸ Nenhum snapshot novo.")

    # Aguarda antes do próximo ciclo
    if ciclo < MAX_CICLOS:
        time.sleep(INTERVALO_SEG)

print("\n" + "=" * 55)
print("   POLLING ENCERRADO")
print(f"   Total processados: {len(snapshots_processados)}")
print("=" * 55)